# Dataset exploration

Quick visual sanity check of the **Mendeley pothole video dataset**:

* clip counts per split,
* duration / resolution stats for a sample,
* a side-by-side preview of a single (rgb, mask) frame pair,
* a histogram of per-frame mask area (in pixels) to understand bbox sizes.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('..').resolve().parent  # repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2, numpy as np, matplotlib.pyplot as plt
from sfm_yolo.src.utils.data_loader import MendeleyVideoDataset, mask_to_bboxes

DATA_ROOT = ROOT / 'data' / 'pothole_video' / 'pothole_video'
print('Dataset root:', DATA_ROOT, 'exists =', DATA_ROOT.exists())

In [ ]:
for split in ('train', 'val', 'test'):
    ds = MendeleyVideoDataset(DATA_ROOT, split=split, frame_stride=1)
    print(f'{split:5s}: {len(ds)} clip pairs')

In [ ]:
ds = MendeleyVideoDataset(DATA_ROOT, split='train', frame_stride=10)
frame = next(ds.iter_clip_frames(0))
boxes = mask_to_bboxes(frame.mask, min_pixels=80)

vis = frame.rgb.copy()
for x1, y1, x2, y2 in boxes:
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(cv2.cvtColor(frame.rgb, cv2.COLOR_BGR2RGB)); ax[0].set_title('RGB')
ax[1].imshow(frame.mask, cmap='gray'); ax[1].set_title('Mask')
ax[2].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax[2].set_title(f'{len(boxes)} bbox(es)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Histogram of mask areas across the first 30 train clips (every 5th frame)
ds = MendeleyVideoDataset(DATA_ROOT, split='train', frame_stride=5)
areas = []
for ci in range(min(30, len(ds))):
    for f in ds.iter_clip_frames(ci):
        for x1, y1, x2, y2 in mask_to_bboxes(f.mask, min_pixels=20):
            areas.append((x2 - x1) * (y2 - y1))
print(f'Boxes collected: {len(areas)}')
if areas:
    plt.hist(np.log10(np.asarray(areas)), bins=40)
    plt.xlabel('log10(bbox area)'); plt.ylabel('count'); plt.title('bbox area histogram')
    plt.show()